In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import sys
sys.path.append(r"D:/projects/FedLGR_SocRob")
from models.MobileNet import Net

model = Net().to(DEVICE)

In [ ]:
import sys
sys.path.append('..')
from data_processing.data_utils import get_domain_dataloaders_from_hdf5

domains = ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']
cols = ['image_path']

In [ ]:
def infer_per_domain(model, device, domain_dataloaders, eval_subset):
    domain_data = {}
    model.eval()
    with torch.no_grad():
        for domain, loaders in domain_dataloaders.items():
            domain_preds = []
            domain_labels = []
            for batch in loaders[eval_subset]:
                inputs, labels, _ = batch
                inputs = tuple(x.to(device) for x in inputs)
                labels = labels.to(device)

                outputs = model(*inputs)

                domain_preds.append(outputs.detach())
                domain_labels.append(labels.detach())

            domain_preds = torch.vstack(domain_preds).cpu()
            domain_labels = torch.vstack(domain_labels).cpu()
            
            domain_data[domain] = (domain_preds, domain_labels)
    
    return domain_data

In [ ]:
model_results = {}

for fold_num in range(5):

    model_path = f"D:/projects/FedLGR_SocRob/Output/fold{fold_num}/3/FedRoot/LGR/fullmod0.pth"
    state = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state, strict=True)
    model.eval()

    dataloader = get_domain_dataloaders_from_hdf5(hdf5_path=f"../data/mean_data_pepper_fold{fold_num}.hdf5", domains=domains, img_path_cols=cols)
    domain_test_preds_targets = infer_per_domain(model, DEVICE, dataloader, 'test')

    model_results[f'FedLGR_fold={fold_num}'] = {"domain_test_preds_targets": domain_test_preds_targets}

In [ ]:
import pickle
with open('./results/FedLGR_domain_preds_targets_folds.pkl', 'wb') as f:
    pickle.dump(model_results, f)